# Wizualizacja Uprawnień Looker - Wykres Sankey

Poniższy kod ładuje plik `sankey_data.json` i generuje piękny, interaktywny wykres przepływowy za pomocą biblioteki `Plotly`. 
Wykres prezentuje ścieżkę:
`Model -> Explore -> Dashboard -> Group -> User`

*Aby wykres działał, upewnij się że uruchomiłeś skrypty `extract_raw_data.py` oraz `build_sankey_data.py`.*

In [ ]:
# Wygeneruj sankey_data.json bez wychodzenia z Notatnika!
from build_sankey_data import build_sankey

build_sankey(
    input_file="permissions_looker_data.json",
    output_file="sankey_data.json",
    target_type="user",          # 'user' | 'group' | 'role'
    target_models=None,           # np. ["model_a", "model_b"]
    target_entities=None,         # np. ["jan.kowalski@firma.pl"] lub ["Nazwa Grupy"]
    target_explores=None,         # np. ["orders", "sessions"]
    target_dashboards=None        # np. ["Sales Overview"]
)


In [ ]:
import json
from IPython.display import display

# Proba uzycia FigureWidget (wymaga ipywidgets)
try:
    import plotly.graph_objects as go
    from plotly.graph_objects import FigureWidget
    USE_WIDGET = True
except Exception:
    USE_WIDGET = False

# 1. Wczytanie danych
with open('sankey_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

nodes = data.get('nodes', [])
links = data.get('links', [])

if not nodes or not links:
    print('Brak danych! Uruchom komórkę build_sankey powyżej.')
else:
    TYPE_COLORS = {
        'model':     'rgba(231, 76, 60, 0.9)',
        'explore':   'rgba(230, 126, 34, 0.9)',
        'dashboard': 'rgba(52, 152, 219, 0.9)',
        'group':     'rgba(155, 89, 182, 0.9)',
        'role':      'rgba(26, 188, 156, 0.9)',
        'user':      'rgba(52, 73, 94, 0.9)',
    }
    DIMMED_NODE = 'rgba(200, 200, 200, 0.12)'
    DIMMED_LINK = 'rgba(200, 200, 200, 0.04)'
    ACTIVE_LINK = 'rgba(255, 200, 0, 0.5)'

    node_base_colors = [TYPE_COLORS.get(n.get('type'), 'rgba(150,150,150,0.8)') for n in nodes]
    link_base_colors = [ACTIVE_LINK for _ in links]

    sankey_trace = go.Sankey(
        arrangement='snap',
        node=dict(
            pad=20,
            thickness=28,
            line=dict(color='white', width=0.5),
            label=[n.get('label', '') for n in nodes],
            color=node_base_colors,
            hovertemplate='<b>%{label}</b><extra></extra>',
        ),
        link=dict(
            source=[l['source'] for l in links],
            target=[l['target'] for l in links],
            value=[l.get('value', 1) for l in links],
            color=link_base_colors,
        )
    )

    layout = go.Layout(
        title_text='Przepływ Uprawnień: Model → Explore → Dashboard → Encja',
        font=dict(size=12, family='Inter, Arial, sans-serif', color='white'),
        height=820,
        paper_bgcolor='#1a1a2e',
        margin=dict(l=20, r=20, t=60, b=20)
    )

    if USE_WIDGET:
        fig = FigureWidget(data=[sankey_trace], layout=layout)

        def on_click(trace, points, state):
            if not points.point_inds:
                return
            clicked = points.point_inds[0]
            conn_links = set()
            conn_nodes = {clicked}
            for i, l in enumerate(links):
                if l['source'] == clicked or l['target'] == clicked:
                    conn_links.add(i)
                    conn_nodes.add(l['source'])
                    conn_nodes.add(l['target'])
            with fig.batch_update():
                fig.data[0].node.color = [
                    node_base_colors[i] if i in conn_nodes else DIMMED_NODE
                    for i in range(len(nodes))
                ]
                fig.data[0].link.color = [
                    ACTIVE_LINK if i in conn_links else DIMMED_LINK
                    for i in range(len(links))
                ]

        def on_dblclick(trace, points, state):
            with fig.batch_update():
                fig.data[0].node.color = node_base_colors
                fig.data[0].link.color = link_base_colors

        fig.data[0].on_click(on_click)
        fig.data[0].on_click(on_dblclick)
        print('💡 Kliknij węzeł, aby wyróżnić połączone ścieżki.')
        display(fig)
    else:
        # Fallback - zwykly Figure
        fig = go.Figure(data=[sankey_trace], layout=layout)
        fig.show()
